## ML_1040: Cosine LR Scheduler with Warmup

**Difficulty**: Medium | **TorchCode**: #30

### Problem
Implement a learning rate schedule with linear warmup followed by cosine annealing. Used in GPT, BERT, and most modern LLM training.

### Formula
$$\text{lr}(t) = \begin{cases} \alpha_{\max} \cdot \dfrac{t}{t_{\text{warm}}} & t < t_{\text{warm}} \\[6pt] \alpha_{\min} + \dfrac{\alpha_{\max}-\alpha_{\min}}{2}\left(1 + \cos\!\left(\pi \cdot \dfrac{t - t_{\text{warm}}}{T - t_{\text{warm}}}\right)\right) & t \ge t_{\text{warm}} \end{cases}$$

In [ ]:
import math

def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step >= total_steps:
        return min_lr
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

In [ ]:
import math

# --- Test 1: Start of warmup ---
lr = cosine_lr_schedule(step=0, total_steps=100, warmup_steps=10, max_lr=0.001, min_lr=0.0)
assert abs(lr) < 1e-8

# --- Test 2: End of warmup ---
lr = cosine_lr_schedule(step=10, total_steps=100, warmup_steps=10, max_lr=0.001)
assert abs(lr - 0.001) < 1e-8

# --- Test 3: End of schedule ---
lr = cosine_lr_schedule(step=100, total_steps=100, warmup_steps=10, max_lr=0.001, min_lr=0.0001)
assert abs(lr - 0.0001) < 1e-6

# --- Test 4: Warmup is monotonically increasing ---
lrs = [cosine_lr_schedule(i, 100, 10, 0.001) for i in range(11)]
for i in range(len(lrs) - 1):
    assert lrs[i] <= lrs[i+1] + 1e-10

# --- Test 5: Cosine shape midpoint ---
lr = cosine_lr_schedule(step=55, total_steps=100, warmup_steps=10, max_lr=0.001, min_lr=0.0)
progress = (55 - 10) / (100 - 10)
expected = 0.5 * 0.001 * (1 + math.cos(math.pi * progress))
assert abs(lr - expected) < 1e-8

print('All tests passed!')